In [ ]:

!pip install fastapi uvicorn nest-asyncio

/usr/lib/python3.12/pathlib.py:404: RuntimeWarning: coroutine 'Server.serve' was never awaited
  parsed = [sys.intern(str(x)) for x in rel.split(sep) if x and x != '.']


In [ ]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 6.3 MB/s eta 0:00:00


<frozen posixpath>:82: RuntimeWarning: coroutine 'Server.serve' was never awaited


In [ ]:
from groq import Groq
from google.colab import userdata
import json
import csv

client_groq = Groq(api_key=userdata.get('GROQ_API_KEY'))
client = Groq(api_key=userdata.get('GROQ_API_KEY'))

In [ ]:
def analyse_customer_structured(client, customer):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": """You are a banking risk analyst.
                Always respond with ONLY valid JSON, no other text.
                Use exactly this structure:
                {
                    "customer_name": "string",
                    "risk_score": number between 1-10,
                    "risk_level": "Low" or "Medium" or "High",
                    "reason": "one sentence explanation",
                    "recommendation": "one sentence action"
                }"""
            },
            {
                "role": "user",
                "content": f"""Analyse this customer:
                Name: {customer['name']}
                Balance: € {customer['balance']}
                Transactions: {customer['transactions']}
                Active: {customer['is_active']}"""
            }
        ]
    )
    return json.loads(response.choices[0].message.content)




In [ ]:
from groq import Groq
import json
from google.colab import userdata



def generate_user_story(requirement):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": """You are a senior AI Product Owner with
                10 years experience in banking and financial services.

                When given a business requirement, respond with ONLY
                valid JSON in exactly this structure:
                {
                    "user_story": "As a [role] I want [feature] so that [benefit]",
                    "acceptance_criteria": [
                        "Given [context] When [action] Then [outcome]",
                        "Given [context] When [action] Then [outcome]",
                        "Given [context] When [action] Then [outcome]"
                    ],
                    "definition_of_done": [
                        "criterion 1",
                        "criterion 2",
                        "criterion 3"
                    ],
                    "story_points": number,
                    "priority": "High" or "Medium" or "Low"
                }"""
            },
            {
                "role": "user",
                "content": f"Generate a user story for this requirement: {requirement}"
            }
        ]
    )
    return json.loads(response.choices[0].message.content)



In [ ]:
from groq import Groq
from google.colab import userdata
import json


def check_eu_ai_act(use_case):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": """You are an EU AI Act compliance expert
                specialising in financial services.

                Analyse AI use cases and classify them under the EU AI Act.

                Respond with ONLY valid JSON in exactly this structure:
                {
                    "use_case_summary": "brief summary of the use case",
                    "risk_tier": "Unacceptable" or "High Risk" or "Limited Risk" or "Minimal Risk",
                    "risk_tier_reason": "why this tier applies",
                    "compliance_requirements": [
                        "requirement 1",
                        "requirement 2",
                        "requirement 3"
                    ],
                    "red_flags": [
                        "red flag 1",
                        "red flag 2"
                    ],
                    "recommended_actions": [
                        "action 1",
                        "action 2",
                        "action 3"
                    ],
                    "compliant_by": "date by which compliance is required"
                }"""
            },
            {
                "role": "user",
                "content": f"Analyse this AI use case for EU AI Act compliance: {use_case}"
            }
        ]
    )
    return json.loads(response.choices[0].message.content)



In [ ]:
def assess_taxonomy(client):

    if not client['nace_code']:
        return {
            "client_id": client['client_id'],
            "client_name": client['client_name'],
            "assessment_tier": "CANNOT ASSESS",
            "taxonomy_eligible": None,
            "taxonomy_aligned": None,
            "substantial_contribution": None,
            "dnsh_assessment": None,
            "minimum_safeguards": None,
            "revenue_kpi": None,
            "capex_kpi": None,
            "gaps": ["Insufficient data for any assessment"],
            "recommendations": ["Collect NACE code, sector, GHG data and safeguards confirmation from client"]
        }

    # Build context
    capex_display = f"€ {client['capex_eur']:,}" if client['capex_eur'] else 'Not provided'

    context = f"""
    Client: {client['client_name']}
    Sector: {client['sector'] or 'Unknown'}
    NACE Code: {client['nace_code']}
    Main Purpose: {client['main_purpose'] or 'Not provided'}
    Sub Purpose: {client['sub_purpose'] or 'Not provided'}
    Product Type: {client['product_type']}
    Reporting Year: {client['reporting_year']}
    Substantial Contribution Objective: {client['substantial_contribution_objective'] or 'Not provided'}
    Revenue Eligible %: {client['revenue_eligible_pct'] or 'Not provided'}
    CapEx Eligible %: {client['capex_eligible_pct'] or 'Not provided'}
    GHG Scope 1+2 (tonnes): {client['ghg_scope1_2_tonnes'] or 'Not provided'}
    Minimum Safeguards Compliant: {client['minimum_safeguards_compliant']}
    Annual Turnover: € {client['annual_turnover_eur']:,}
    CapEx: {capex_display}
    """

    if client['nace_code'] and client['main_purpose'] and client['sub_purpose'] and client['ghg_scope1_2_tonnes']:
        instruction = "Perform a complete 4-step EU Taxonomy assessment."
    elif client['nace_code'] and client['main_purpose']:
        instruction = "Perform a partial EU Taxonomy assessment. Check eligibility and substantial contribution only."
    else:
        instruction = "Perform an eligibility check only based on NACE code."

    response = client_groq.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": """You are a senior EU Taxonomy compliance expert at a Dutch bank.
You follow the official EU Taxonomy Compass 4-step framework strictly.

CRITICAL RULES:
- Diesel or fossil fuel transport is NOT taxonomy aligned
- Traditional blast furnace steel is NOT taxonomy aligned
- Only activities meeting Technical Screening Criteria are aligned
- Be conservative — when in doubt mark as not aligned

Respond with ONLY valid JSON. No preamble, no markdown.

                Use exactly this structure:
                {
                    "assessment_tier": "FULL" or "PARTIAL" or "ELIGIBILITY",
                    "taxonomy_eligible": true or false,
                    "taxonomy_aligned": true or false or null,
                    "substantial_contribution": {
                        "objective": "objective name",
                        "meets_criteria": true or false,
                        "reasoning": "one sentence"
                    },
                    "dnsh_assessment": {
                        "climate_mitigation": "Pass" or "Fail" or "Incomplete",
                        "climate_adaptation": "Pass" or "Fail" or "Incomplete",
                        "water": "Pass" or "Fail" or "Incomplete",
                        "circular_economy": "Pass" or "Fail" or "Incomplete",
                        "pollution": "Pass" or "Fail" or "Incomplete",
                        "biodiversity": "Pass" or "Fail" or "Incomplete",
                        "overall": "Pass" or "Fail" or "Incomplete"
                    },
                    "minimum_safeguards": "Pass" or "Fail" or "Incomplete",
                    "revenue_kpi": number or null,
                    "capex_kpi": number or null,
                    "gaps": ["gap 1", "gap 2"],
                    "recommendations": ["action 1", "action 2"]
                }"""
            },
            {
                "role": "user",
                "content": f"{instruction}\n\nClient data:\n{context}"
            }
        ]
    )

    # Debug — see raw response
    raw = response.choices[0].message.content
    print(f"Raw response: {raw[:200]}")

    # Clean and parse
    try:
        clean = raw.strip()
        if clean.startswith("```"):
            clean = clean.split("```")[1]
            if clean.startswith("json"):
                clean = clean[4:]
        result = json.loads(clean.strip())
        result['client_id'] = client['client_id']
        result['client_name'] = client['client_name']
        return result
    except Exception as e:
        print(f"Parse error: {e}")
        print(f"Full raw response: {raw}")
        return None


In [ ]:
from fastapi import FastAPI
import nest_asyncio
import uvicorn
import threading

nest_asyncio.apply()

app = FastAPI(title="AI Banking Platform", version="1.0")

@app.get("/")
def home():
    return {
        "message": "Welcome to AI Banking Platform",
        "tools": [
            "risk-assessment",
            "user-stories",
            "eu-ai-act",
            "eu-taxonomy"
        ]
    }

# Run in a separate thread so Colab doesn't block
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

thread = threading.Thread(target=run_server)
thread.daemon = True
thread.start()

print("✅ Server running on port 8000")

✅ Server running on port 8000


In [ ]:
from pydantic import BaseModel

# Define what the request should look like
class CustomerInput(BaseModel):
    name: str
    balance: float
    transactions: int
    is_active: bool

@app.post("/risk-assessment")
def risk_assessment(customer: CustomerInput):
    result = analyse_customer_structured(client_groq, {
        "name": customer.name,
        "balance": customer.balance,
        "transactions": customer.transactions,
        "is_active": customer.is_active
    })
    return result

In [ ]:
import requests

response = requests.post(
    "http://localhost:8000/risk-assessment",
    json={
        "name": "Jan de Vries",
        "balance": 750,
        "transactions": 142,
        "is_active": True
    }
)
print(response.json())

INFO:     127.0.0.1:42844 - "POST /risk-assessment HTTP/1.1" 200 OK
{'customer_name': 'Jan de Vries', 'risk_score': 4, 'risk_level': 'Low', 'reason': 'The customer has a moderate balance and a relatively high number of transactions, indicating a stable financial activity.', 'recommendation': "Continue monitoring the customer's account activity to ensure the risk level remains stable."}


In [ ]:
from pydantic import BaseModel
from typing import Optional

# User Stories
class RequirementInput(BaseModel):
    requirement: str

@app.post("/user-stories")
def user_stories(input: RequirementInput):
    result = generate_user_story(input.requirement)
    return result

# EU AI Act
class UseCaseInput(BaseModel):
    use_case: str

@app.post("/eu-ai-act")
def eu_ai_act(input: UseCaseInput):
    result = check_eu_ai_act(input.use_case)
    return result

# EU Taxonomy
class TaxonomyInput(BaseModel):
    client_id: str
    client_name: str
    sector: Optional[str] = None
    nace_code: Optional[str] = None
    main_purpose: Optional[str] = None
    sub_purpose: Optional[str] = None
    loan_amount_eur: float
    ghg_scope1_2_tonnes: Optional[float] = None

@app.post("/eu-taxonomy")
def eu_taxonomy(input: TaxonomyInput):
    client = {
        "client_id": input.client_id,
        "client_name": input.client_name,
        "sector": input.sector,
        "nace_code": input.nace_code,
        "main_purpose": input.main_purpose,
        "sub_purpose": input.sub_purpose,
        "product_type": "Term Loan",
        "country": "Netherlands",
        "annual_turnover_eur": 0,
        "capex_eur": 0,
        "loan_amount_eur": input.loan_amount_eur,
        "reporting_year": 2024,
        "substantial_contribution_objective": "Climate Change Mitigation",
        "revenue_eligible_pct": None,
        "capex_eligible_pct": None,
        "ghg_scope1_2_tonnes": input.ghg_scope1_2_tonnes,
        "minimum_safeguards_compliant": True
    }
    result = assess_taxonomy(client)
    return result

In [ ]:
# Test User Stories
r1 = requests.post("http://localhost:8000/user-stories",
    json={"requirement": "Compliance officers need AML screening"})
print("✅ User Stories:", r1.json()['user_story'][:50])

# Test EU AI Act
r2 = requests.post("http://localhost:8000/eu-ai-act",
    json={"use_case": "AI credit scoring system for retail bank"})
print("✅ EU AI Act:", r2.json()['risk_tier'])

# Test EU Taxonomy
r3 = requests.post("http://localhost:8000/eu-taxonomy",
    json={
        "client_id": "NL-001",
        "client_name": "GreenHome Mortgages BV",
        "sector": "Real Estate",
        "nace_code": "F41.1",
        "main_purpose": "Construction",
        "sub_purpose": "Green Buildings",
        "loan_amount_eur": 8000000,
        "ghg_scope1_2_tonnes": 450
    })
print("✅ EU Taxonomy:", r3.json()['taxonomy_aligned'])

INFO:     127.0.0.1:46948 - "POST /user-stories HTTP/1.1" 200 OK
✅ User Stories: As a compliance officer I want to perform AML scre
INFO:     127.0.0.1:46950 - "POST /eu-ai-act HTTP/1.1" 200 OK
✅ EU AI Act: High Risk
Raw response: ```
{
    "assessment_tier": "FULL",
    "taxonomy_eligible": true,
    "taxonomy_aligned": null,
    "substantial_contribution": {
        "objective": "Climate Change Mitigation",
        "meets_cri
INFO:     127.0.0.1:46964 - "POST /eu-taxonomy HTTP/1.1" 200 OK
✅ EU Taxonomy: None


In [ ]:
print("📚 API Docs available at:")
print("http://localhost:8000/docs")


📚 API Docs available at:
http://localhost:8000/docs


In [ ]:
response = requests.get("http://localhost:8000/docs")
print(f"Docs status: {response.status_code}")

INFO:     127.0.0.1:45650 - "GET /docs HTTP/1.1" 200 OK
Docs status: 200
